In [3]:
import torch
import torch.nn as nn
import random

In [ ]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_size):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim)
    self.gru = nn.GRU(embed_dim, hidden_size)

  def forward(self, x):
    out, hn = self.gru(self.embed(x))

    return hn

class Decoder(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_size):
    super().__init__()
    self.vocab_size = vocab_size
    self.embed = nn.Embedding(vocab_size, embed_dim)
    self.gru = nn.GRU(embed_dim, hidden_size)
    self.lin = nn.Linear(hidden_size, vocab_size)

  def forward_step(self, input_token, hidden):
    embeded = self.embed(input_token).unsqueeze(0)

    out, hidden = self.gru(embeded, hidden)

    logits = self.lin(out.squeeze(0))

    return logits,hidden


In [4]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder, teacher_forcing=0.5):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.teacher_forcing = teacher_forcing

  def forward(self, x, target):
    hidden = self.encoder(x)

    target_len, batch_size = target.shape

    decoder_input = target[0]

    outputs = torch.zeros(target_len, batch_size, self.decoder.vocab_size)

    for t in range(1, target_len):
      logits, hidden = self.decoder.forward_step(decoder_input, hidden)

      predicted = torch.argmax(logits, dim=1)

      outputs[t] = logits

      if random.uniform(0.1, 1.0) < self.teacher_forcing:
        decoder_input = target[t]

      else:
        decoder_input = predicted

    return outputs